In [1]:
import pandas as pd

# Load the training and test datasets
train_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/DSEval/datasets/05_patient_profile/train.csv'
test_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/DSEval/datasets/05_patient_profile/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Display the first few rows of the datasets to understand their structure
print("Training Data:")
print(train_df.head())
print("\nTest Data:")
print(test_df.head())


Training Data:
                     Disease Fever  ... Cholesterol Level Outcome Variable
0                Ebola Virus   Yes  ...            Normal         Positive
1  Conjunctivitis (Pink Eye)    No  ...              High         Positive
2               Pancreatitis    No  ...              High         Positive
3               Pancreatitis   Yes  ...            Normal         Negative
4            Hyperthyroidism   Yes  ...            Normal         Negative

[5 rows x 10 columns]

Test Data:
                                        Disease  ... Outcome Variable
0                                Hypothyroidism  ...         Positive
1                                   Tonsillitis  ...         Positive
2  Chronic Obstructive Pulmonary Disease (COPD)  ...         Positive
3                                  Lyme Disease  ...         Positive
4                                      Diabetes  ...         Positive

[5 rows x 10 columns]


In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Check column information for the training data
column_info_train = get_column_info(train_df)
print("Column Information for Training Data:")
print(column_info_train)

# Check column information for the test data
column_info_test = get_column_info(test_df)
print("\nColumn Information for Test Data:")
print(column_info_test)


2025-09-10 08:07:49.083 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


Column Information for Training Data:
{'Category': ['Disease', 'Fever', 'Cough', 'Fatigue', 'Difficulty Breathing', 'Gender', 'Blood Pressure', 'Cholesterol Level', 'Outcome Variable'], 'Numeric': ['Age'], 'Datetime': [], 'Others': []}

Column Information for Test Data:
{'Category': ['Disease', 'Fever', 'Cough', 'Fatigue', 'Difficulty Breathing', 'Gender', 'Blood Pressure', 'Cholesterol Level', 'Outcome Variable'], 'Numeric': ['Age'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import OneHotEncode

# Define the categorical features to be encoded, excluding 'Disease'
categorical_features = ['Fever', 'Cough', 'Fatigue', 'Difficulty Breathing', 'Gender', 'Blood Pressure', 'Cholesterol Level']

# Initialize the OneHotEncode tool
encoder = OneHotEncode(features=categorical_features)

# Fit and transform the training data
train_df_encoded = encoder.fit_transform(train_df.copy())

# Transform the test data using the same encoder
test_df_encoded = encoder.transform(test_df.copy())

# Display the first few rows of the encoded training and test data
print("Encoded Training Data:")
print(train_df_encoded.head())
print("\nEncoded Test Data:")
print(test_df_encoded.head())


Encoded Training Data:
                     Disease  ...  Cholesterol Level_Normal
0                Ebola Virus  ...                       1.0
1  Conjunctivitis (Pink Eye)  ...                       0.0
2               Pancreatitis  ...                       0.0
3               Pancreatitis  ...                       1.0
4            Hyperthyroidism  ...                       1.0

[5 rows x 19 columns]

Encoded Test Data:
                                        Disease  ...  Cholesterol Level_Normal
0                                Hypothyroidism  ...                       0.0
1                                   Tonsillitis  ...                       0.0
2  Chronic Obstructive Pulmonary Disease (COPD)  ...                       0.0
3                                  Lyme Disease  ...                       0.0
4                                      Diabetes  ...                       1.0

[5 rows x 19 columns]


D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Check column information for the encoded training data
column_info_train = get_column_info(train_df_encoded)
print("Column Information for Training Data:")
print(column_info_train)

# Check column information for the encoded test data
column_info_test = get_column_info(test_df_encoded)
print("\nColumn Information for Test Data:")
print(column_info_test)


Column Information for Training Data:
{'Category': ['Disease', 'Outcome Variable'], 'Numeric': ['Age', 'Fever_No', 'Fever_Yes', 'Cough_No', 'Cough_Yes', 'Fatigue_No', 'Fatigue_Yes', 'Difficulty Breathing_No', 'Difficulty Breathing_Yes', 'Gender_Female', 'Gender_Male', 'Blood Pressure_High', 'Blood Pressure_Low', 'Blood Pressure_Normal', 'Cholesterol Level_High', 'Cholesterol Level_Low', 'Cholesterol Level_Normal'], 'Datetime': [], 'Others': []}

Column Information for Test Data:
{'Category': ['Disease', 'Outcome Variable'], 'Numeric': ['Age', 'Fever_No', 'Fever_Yes', 'Cough_No', 'Cough_Yes', 'Fatigue_No', 'Fatigue_Yes', 'Difficulty Breathing_No', 'Difficulty Breathing_Yes', 'Gender_Female', 'Gender_Male', 'Blood Pressure_High', 'Blood Pressure_Low', 'Blood Pressure_Normal', 'Cholesterol Level_High', 'Cholesterol Level_Low', 'Cholesterol Level_Normal'], 'Datetime': [], 'Others': []}


In [5]:
from metagpt.tools.libs.feature_engineering import GeneralSelection, VarianceBasedSelection
from sklearn.feature_selection import SelectKBest, f_classif

# Initialize feature selection tools
general_selection = GeneralSelection(label_col='Outcome Variable')
variance_selection = VarianceBasedSelection(label_col='Outcome Variable')

# Apply GeneralSelection to remove features with all NaNs or only one unique value
train_df_selected = general_selection.fit_transform(train_df_encoded.copy())
test_df_selected = general_selection.transform(test_df_encoded.copy())

# Apply VarianceBasedSelection to remove features with low variance
train_df_selected = variance_selection.fit_transform(train_df_selected.copy())
test_df_selected = variance_selection.transform(test_df_selected.copy())

# Use SelectKBest to select the top 5 features
selector = SelectKBest(score_func=f_classif, k=5)
X_train_selected = selector.fit_transform(train_df_selected.drop(columns=['Outcome Variable']), train_df_selected['Outcome Variable'])
X_test_selected = selector.transform(test_df_selected.drop(columns=['Outcome Variable']))

# Get the selected feature names
selected_features = train_df_selected.drop(columns=['Outcome Variable']).columns[selector.get_support()]

print("Selected Features:", selected_features)


Selected Features: Index(['Gender_Female', 'Gender_Male', 'Blood Pressure_High',
       'Cholesterol Level_High', 'Cholesterol Level_Normal'],
      dtype='object')


In [6]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Check the latest DataFrame after feature selection
df = test_df_selected

column_info = get_column_info(df)
print("column_info")
print(column_info)


column_info
{'Category': ['Outcome Variable'], 'Numeric': ['Age', 'Fever_No', 'Fever_Yes', 'Cough_No', 'Cough_Yes', 'Fatigue_No', 'Fatigue_Yes', 'Difficulty Breathing_No', 'Difficulty Breathing_Yes', 'Gender_Female', 'Gender_Male', 'Blood Pressure_High', 'Blood Pressure_Low', 'Blood Pressure_Normal', 'Cholesterol Level_High', 'Cholesterol Level_Low', 'Cholesterol Level_Normal'], 'Datetime': [], 'Others': []}


In [7]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Assuming train_df_encoded and test_df_encoded are already defined from previous tasks
X_train = train_df_encoded.drop(columns=['Outcome Variable'])
y_train = train_df_encoded['Outcome Variable']
X_test = test_df_encoded.drop(columns=['Outcome Variable'])
y_test = test_df_encoded['Outcome Variable']

# Build logistic regression model1 using all features
model1 = LogisticRegression(max_iter=1000)
model1.fit(X_train, y_train)

# Predict on the test set
y_pred_model1 = model1.predict(X_test)

# Evaluate the performance of model1
accuracy_model1 = accuracy_score(y_test, y_pred_model1)
conf_matrix_model1 = confusion_matrix(y_test, y_pred_model1)
class_report_model1 = classification_report(y_test, y_pred_model1)

print("Model 1 - Logistic Regression with All Features")
print(f"Accuracy: {accuracy_model1}")
print("Confusion Matrix:")
print(conf_matrix_model1)
print("Classification Report:")
print(class_report_model1)


ValueError: could not convert string to float: 'Ebola Virus'